In [ ]:
from moabb.paradigms import LeftRightImagery
import time
from moabb.datasets import BNCI2014_001

dataset = BNCI2014_001()
subjects = [1]  # start with one subject only
data = dataset.get_data(subjects=subjects)
print(type(data))
print(data.keys())

paradigm = LeftRightImagery(fmin=8, fmax=35)

t0 = time.time()
X, y, metadata = paradigm.get_data(dataset=dataset, subjects=[1])
print("get_data took", time.time() - t0, "seconds")

print("X shape:", X.shape)
print("y shape:", y.shape)
print("metadata shape:", metadata.shape)
print("unique labels:", set(y))

In [ ]:
import numpy as np
import torch
import random

seed = 42

np.random.seed(seed)
torch.manual_seed(seed)
random.seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

In [ ]:
label_map = {"left_hand": 0, "right_hand": 1}
y_num = np.array([label_map[label] for label in y], dtype=np.int64)

print("y_num shape:", y_num.shape)
print("class counts:", {0: int((y_num == 0).sum()), 1: int((y_num == 1).sum())})

In [ ]:
# Add EEGNet input dimension: (N, 1, channels, time)
X_eegnet = X[:, None, :, :]

X_tensor = torch.tensor(X_eegnet, dtype=torch.float32)
y_tensor = torch.tensor(y_num, dtype=torch.long)

print("X_tensor shape:", X_tensor.shape)
print("y_tensor shape:", y_tensor.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_tensor.numpy(),
    y_tensor.numpy(),
    test_size=0.2,
    random_state=42,
    stratify=y_tensor.numpy()
)

print("train:", X_train.shape, y_train.shape)
print("val:", X_val.shape, y_val.shape)

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)

In [ ]:
# Compute mean/std from training set only
train_mean = X_train.mean(dim=(0, 3), keepdim=True)
train_std = X_train.std(dim=(0, 3), keepdim=True) + 1e-6

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

batch_size = 32

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

g = torch.Generator()
g.manual_seed(42)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=g)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

xb, yb = next(iter(train_loader))
print("batch X:", xb.shape)
print("batch y:", yb.shape)

In [ ]:
from torch import device
import torch.nn as nn
class EEGNet(nn.Module):
    def __init__(self, n_classes=2, dropout=0.25):
        super().__init__()

        # Block 1: temporal convolution
        self.block1 = nn.Sequential(
            nn.Conv2d(
                in_channels=1,
                out_channels=8,
                kernel_size=(1, 32),
                padding=(0, 16),
                bias=False
            ),
            nn.BatchNorm2d(8),

            # spatial convolution across channels
            nn.Conv2d(
                in_channels=8,
                out_channels=16,
                kernel_size=(22, 1),
                groups=8,
                bias=False
            ),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout)
        )

        # Block 2: more temporal feature extraction
        self.block2 = nn.Sequential(
            nn.Conv2d(
                in_channels=16,
                out_channels=16,
                kernel_size=(1, 16),
                padding=(0, 8),
                groups=16,
                bias=False
            ),
            nn.Conv2d(
                in_channels=16,
                out_channels=16,
                kernel_size=(1, 1),
                bias=False
            ),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(dropout)
        )

        # We'll infer classifier input size dynamically
        self.classifier = nn.Linear(496, 2)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = torch.flatten(x, start_dim=1)

        x = self.classifier(x)
        return x

model = EEGNet()

xb, yb = next(iter(train_loader))
out = model(xb)

print("Input shape:", xb.shape)
print("Output shape:", out.shape)
print(model)

In [ ]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = EEGNet().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)
num_epochs = 50

In [ ]:
best_val_acc = 0.0
best_epoch = 0
best_state = None

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0
    train_correct = 0
    train_total = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * xb.size(0)
        preds = torch.argmax(outputs, dim=1)
        train_correct += (preds == yb).sum().item()
        train_total += yb.size(0)

    epoch_train_loss = running_train_loss / train_total
    epoch_train_acc = train_correct / train_total

    model.eval()
    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)

            outputs = model(xb)
            loss = criterion(outputs, yb)

            running_val_loss += loss.item() * xb.size(0)
            preds = torch.argmax(outputs, dim=1)
            val_correct += (preds == yb).sum().item()
            val_total += yb.size(0)

    epoch_val_loss = running_val_loss / val_total
    epoch_val_acc = val_correct / val_total

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    train_accuracies.append(epoch_train_acc)
    val_accuracies.append(epoch_val_acc)

    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        best_epoch = epoch + 1
        best_state = model.state_dict()

    print(
        f"epoch {epoch+1:02d}/{num_epochs} | "
        f"train loss: {epoch_train_loss:.4f} | "
        f"train acc: {epoch_train_acc:.3f} | "
        f"val loss: {epoch_val_loss:.4f} | "
        f"val acc: {epoch_val_acc:.3f}"
    )

print(f"\nbest val acc: {best_val_acc:.3f} at epoch {best_epoch}")

In [ ]:
import os
os.makedirs("saved_models", exist_ok=True)

torch.save(best_state, "saved_models/eegnet_best.pt")
print("saved best model")

best_model = EEGNet().to(device)

# run one dummy batch through to create classifier layer
dummy_x = next(iter(train_loader))[0].to(device)
_ = best_model(dummy_x)

# now load saved weights
best_model.load_state_dict(torch.load("saved_models/eegnet_best.pt", map_location=device))
best_model.eval()

print("best model loaded successfully")

In [ ]:
model = EEGNet()
xb, yb = next(iter(train_loader))
out = model(xb)

print("Input shape:", xb.shape)
print("Output shape:", out.shape)
print(model)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(train_accuracies, label="train accuracy")
plt.plot(val_accuracies, label="val accuracy")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("eegnet training")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="train loss")
plt.plot(val_losses, label="val loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("eegnet loss")
plt.legend()
plt.show()

In [ ]:
import torch.nn.functional as F

def predict_trial(model, trial_tensor, device):
    model.eval()

    with torch.no_grad():
        if trial_tensor.ndim == 3:
            trial_tensor = trial_tensor.unsqueeze(0)  # -> (1, 1, 64, 241)

        trial_tensor = trial_tensor.to(device)
        outputs = model(trial_tensor)
        probs = F.softmax(outputs, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    return pred, probs.cpu().numpy()

In [ ]:
sample_x, sample_y = val_dataset[0]

pred_class, pred_probs = predict_trial(best_model, sample_x, device)

print("True label:", sample_y.item())
print("Predicted class:", pred_class)
print("Probabilities:", pred_probs)

In [ ]:
def get_trial_from_dataset(dataset, idx):
    x, y = dataset[idx]
    return x, y.item()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def run_cursor_trajectory_demo(model, dataset, device, trial_idx=0):
    trial_x, true_label = dataset[trial_idx]
    pred_class, pred_probs = predict_trial(model, trial_x, device)

    # map predicted class to direction
    move_direction = -1 if pred_class == 0 else 1
    confidence = float(pred_probs[0][pred_class])

    # make movement clearly visible
    speed = 0.03 + confidence * 0.05

    print(f"true label: {true_label}")
    print(f"predicted class: {pred_class}")
    print(f"confidence: {confidence:.3f}")
    print(f"move_direction: {move_direction}")
    print(f"speed: {speed:.3f}")

    # start in center
    cursor_x = 0.5
    cursor_y = 0.5

    # target side depends on true label
    target_x = 0.2 if true_label == 0 else 0.8
    target_y = 0.5

    # simulate movement
    xs = [cursor_x]
    ys = [cursor_y]

    for _ in range(20):
        cursor_x += move_direction * speed
        cursor_x = np.clip(cursor_x, 0, 1)
        xs.append(cursor_x)
        ys.append(cursor_y)

    # plot trajectory
    plt.figure(figsize=(8, 3))
    plt.plot(xs, ys, marker='o', label="cursor trajectory")
    plt.scatter([target_x], [target_y], s=300, label="target")
    plt.axvline(0.5, linestyle="--", label="center")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.title("BCI Cursor Trajectory Demo")
    plt.legend()
    plt.show()

    # simple success check
    success = (pred_class == true_label)
    print("Hit correct side?" , success)

run_cursor_trajectory_demo(best_model, val_dataset, device, trial_idx=0)

In [ ]:
def evaluate_cursor_control(model, dataset, device):
    results = []

    for idx in range(len(dataset)):
        trial_x, true_label = dataset[idx]
        pred_class, pred_probs = predict_trial(model, trial_x, device)

        confidence = float(pred_probs[0][pred_class])
        success = int(pred_class == true_label)

        results.append({
            "trial_idx": idx,
            "true_label": int(true_label),
            "pred_class": int(pred_class),
            "confidence": confidence,
            "success": success
        })

    return results

results = evaluate_cursor_control(best_model, val_dataset, device)
print(results[:3])

In [ ]:
import numpy as np

success_rate = np.mean([r["success"] for r in results])
avg_confidence = np.mean([r["confidence"] for r in results])
left_preds = sum(r["pred_class"] == 0 for r in results)
right_preds = sum(r["pred_class"] == 1 for r in results)

print(f"Success rate: {success_rate:.3f}")
print(f"Average confidence: {avg_confidence:.3f}")
print(f"Left predictions: {left_preds}")
print(f"Right predictions: {right_preds}")

In [ ]:
import pandas as pd

df_results = pd.DataFrame(results)
df_results

In [ ]:
best_model = EEGNet().to(device)

dummy_x = next(iter(train_loader))[0].to(device)
_ = best_model(dummy_x)

best_model.load_state_dict(torch.load("saved_models/eegnet_best.pt", map_location=device))
best_model.eval()

In [ ]:
sample_x, sample_y = val_dataset[0]
pred_class, pred_probs = predict_trial(best_model, sample_x, device)

print("True label:", sample_y.item())
print("Predicted class:", pred_class)
print("Probabilities:", pred_probs)

In [ ]:
def predict_trial(model, trial_tensor, device):
    import torch.nn.functional as F

    model.eval()
    with torch.no_grad():
        if trial_tensor.ndim == 3:
            trial_tensor = trial_tensor.unsqueeze(0)  # (1, 1, C, T)

        trial_tensor = trial_tensor.to(device)
        outputs = model(trial_tensor)
        probs = F.softmax(outputs, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    return pred, probs.cpu().numpy()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def run_cursor_trajectory_demo(model, dataset, device, trial_idx=0):
    trial_x, true_label = dataset[trial_idx]
    pred_class, pred_probs = predict_trial(model, trial_x, device)

    move_direction = -1 if pred_class == 0 else 1
    confidence = float(pred_probs[0][pred_class])

    speed = 0.03 + confidence * 0.05

    print(f"true label: {true_label}")
    print(f"predicted class: {pred_class}")
    print(f"confidence: {confidence:.3f}")
    print(f"move_direction: {move_direction}")
    print(f"speed: {speed:.3f}")

    cursor_x = 0.5
    cursor_y = 0.5

    target_x = 0.2 if true_label == 0 else 0.8
    target_y = 0.5

    xs = [cursor_x]
    ys = [cursor_y]

    for _ in range(20):
        cursor_x += move_direction * speed
        cursor_x = np.clip(cursor_x, 0, 1)
        xs.append(cursor_x)
        ys.append(cursor_y)

    plt.figure(figsize=(8, 3))
    plt.plot(xs, ys, marker='o', label="cursor trajectory")
    plt.scatter([target_x], [target_y], s=300, label="target")
    plt.axvline(0.5, linestyle="--", label="center")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.title("BCI Cursor Trajectory Demo")
    plt.legend()
    plt.show()

    success = (pred_class == true_label)
    print("Hit correct side?", success)

In [ ]:
run_cursor_trajectory_demo(best_model, val_dataset, device, trial_idx=0)
run_cursor_trajectory_demo(best_model, val_dataset, device, trial_idx=1)
run_cursor_trajectory_demo(best_model, val_dataset, device, trial_idx=2)

In [ ]:
def evaluate_cursor_control_with_threshold(model, dataset, device, threshold=0.6):
    results = []

    for idx in range(len(dataset)):
        trial_x, true_label = dataset[idx]
        pred_class, pred_probs = predict_trial(model, trial_x, device)

        confidence = float(pred_probs[0][pred_class])

        # only act if confident
        if confidence < threshold:
            action = None  # no movement
            success = False
        else:
            action = pred_class
            success = (pred_class == int(true_label))

        results.append({
            "trial_idx": idx,
            "true_label": int(true_label),
            "pred_class": int(pred_class),
            "confidence": confidence,
            "action": action,
            "success": success
        })

    return results

In [ ]:
results_thresh = evaluate_cursor_control_with_threshold(best_model, val_dataset, device, threshold=0.6)

In [ ]:
import numpy as np

valid_trials = [r for r in results_thresh if r["action"] is not None]

success_rate = np.mean([r["success"] for r in valid_trials])
coverage = len(valid_trials) / len(results_thresh)

print(f"Success rate (when acting): {success_rate:.3f}")
print(f"Coverage (how often model acts): {coverage:.3f}")

In [ ]:
import numpy as np

y_vals = y_val.numpy()

print("Left true:", np.sum(y_vals == 0))
print("Right true:", np.sum(y_vals == 1))

In [ ]:
df_results["pred_class"].value_counts()

In [ ]:
model.eval()

all_preds = []

with torch.no_grad():
    for xb, _ in val_loader:
        xb = xb.to(device)
        outputs = model(xb)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())

import numpy as np
print("pred counts:", np.bincount(all_preds))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

best_model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        outputs = best_model(xb)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.numpy())

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Left", "Right"],
            yticklabels=["Left", "Right"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()